# MAGNet — Inference Demo

Generate multi-person motion with DFOT and visualize results in 3D.

**Pipeline:** Choose a config &rarr; Run inference &rarr; Visualize

| Config | Task |
|--------|------|
| `dd100.yaml` | Joint multi-person generation |
| `dd100_partner_prediction.yaml` | Predict one person given the other |
| `dd100_turn_taking.yaml` | Asynchronous leader-follower generation |
| `dd100_motion_control.yaml` | Generate person B conditioned on A's motion |
| `dd100_partner_inpainting.yaml` | Inpaint one person given the other |
| `dd100_inbetweening.yaml` | Fill motion between keyframes |
| `meta_trio.yaml` | Joint 3-person generation (Embody3D trio) |


## 1. Select Config

Set `CONFIG` to one of the YAML files listed above.

In [11]:
CONFIG = "configs/inference/dfot/meta_trio.yaml"

In [12]:
from omegaconf import OmegaConf

cfg = OmegaConf.load(CONFIG)
print(f"Task:       {cfg.sampling_cfg.sampling_task}")
print(f"Checkpoint: {cfg.checkpoint_dir}")
print(f"Sequences:  {cfg.inference_data_list}")
print(f"Samples:    {cfg.sampling_cfg.sampling_num}")
print(f"Frames:     {cfg.sampling_cfg.sampling_seq_len}")

Task:       joint
Checkpoint: ./checkpoints/dfot/magnet_embody3d_trio/v0
Sequences:  ['DAYLIFE_Doing_chores_together--VQR525--XAO108--RGM410--pilot--MotionPrior--177213-177627', 'ACTING_Thanksgiving_day--VQR525--XAO108--RGM410--pilot--MotionPrior--113311-122307', 'DAYLIFE_Doing_chores_together--VQR525--XAO108--RGM410--pilot--MotionPrior--177672-186218']
Samples:    10
Frames:     300


## 2. Run DFOT Inference

This calls the inference script which loads the model, generates motion samples, and saves `.npz` files.

In [13]:
!bash scripts/run_dfot_inference.sh {CONFIG}

DFOTNetwork Cond
Loading model from checkpoints/vqvae/magnet_embody3d/v0/checkpoints_300000/model.safetensors
Mode: Relative Canonical Cond
test_data file list: dict_keys(['DAYLIFE_Doing_chores_together--VQR525--XAO108--RGM410--pilot--MotionPrior--177213-177627', 'ACTING_Thanksgiving_day--VQR525--XAO108--RGM410--pilot--MotionPrior--113311-122307', 'DAYLIFE_Doing_chores_together--VQR525--XAO108--RGM410--pilot--MotionPrior--177672-186218'])
DAYLIFE_Doing_chores_together--VQR525--XAO108--RGM410--pilot--MotionPrior--177213-177627
context_seq_len: 32
Sampling:   0%|                                         | 0/801 [00:00<?, ?it/s]cfg_scale_dict: {'clean': 1.0}
[SmoothingGuidance] step=0/801 grad_norm=0.233447 weight=3.0
[FootSkatingGuidance] step=0/801 grad_norm=0.473509 weight=3.0
Sampling:  12%|███▉                            | 97/801 [00:02<00:15, 46.26it/s][SmoothingGuidance] step=100/801 grad_norm=0.136747 weight=3.0
[FootSkatingGuidance] step=100/801 grad_norm=0.348873 weight=3.0
Sampl

## 3. Locate Output Directory

Find the latest inference output (the script timestamps each run).

In [14]:
from pathlib import Path

output_base = Path(cfg.output_dir) / Path(cfg.checkpoint_dir).parents[0].name
output_dirs = sorted(output_base.glob("*"), key=lambda p: p.stat().st_mtime)
OUTPUT_DIR = str(output_dirs[-1])

print(f"Output: {OUTPUT_DIR}")
print(f"Files:")
for f in sorted(Path(OUTPUT_DIR).glob("*.npz")):
    print(f"  {f.name}")

Output: outputs/dfot/magnet_embody3d_trio/250k_test_trio_joint_20260320_080638
Files:
  ACTING_Thanksgiving_day--VQR525--XAO108--RGM410--pilot--MotionPrior--113311-122307.npz
  DAYLIFE_Doing_chores_together--VQR525--XAO108--RGM410--pilot--MotionPrior--177213-177627.npz
  DAYLIFE_Doing_chores_together--VQR525--XAO108--RGM410--pilot--MotionPrior--177672-186218.npz


## 4. Visualize with Viser

Launch an interactive 3D viewer. Open the printed share URL in your browser.

**Interrupt the kernel** to stop the viewer and continue.

In [ ]:
import sys
import viser

viz_dir = str(Path("libs/viz").resolve())
if viz_dir not in sys.path:
    sys.path.insert(0, viz_dir)

from libs.viz.visualizer import load_visualizer, visualize

server = viser.ViserServer(port=8084)
share_url = server.request_share_url()
print(f"Share URL: {share_url}")

try:
    visualize(server, data_root_dir=Path(OUTPUT_DIR))
except KeyboardInterrupt:
    print("Viewer stopped.")

╭────── viser (listening *:8084) ───────╮
│             ╷                         │
│   HTTP      │ http://localhost:8084   │
│   Websocket │ ws://localhost:8084     │
│             ╵                         │
╰───────────────────────────────────────╯

(viser) Share URL requested!

(viser) Generated share URL (expires in 24 hours, max 16 clients): https://cargo-rotate.share.viser.studio

Share URL: https://cargo-rotate.share.viser.studio


(viser) Connection opened (0, 1 total), 63 persistent messages

task mode: joint
updating smpl data
updated smpl data
update visible sample idx list: [0, 1]
task mode: joint
updating smpl data
updated smpl data
task mode: joint
updating smpl data
updated smpl data
